# 🧬 Data Augmentation Pipeline

**Purpose:** Merge Target Dataset with metadata from Source Dataset using Polars (Lazy Mode).

**Environment:** Google Colab (High-RAM preferred, Standard compatible)

---

## Key Features:
- ✅ Memory-efficient loading with `scan_parquet()` (Lazy Mode)
- ✅ Optimized column selection to minimize I/O
- ✅ Automatic data type alignment for joining
- ✅ Feature engineering (gender encoding, insulin type encoding)
- ✅ Works with ~132M row source file on Standard Colab

## 1. Setup & Configuration

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install Polars
!pip install polars pyarrow -q
print("✅ Polars installed successfully!")

In [ ]:
import polars as pl
import gc

print(f"Polars version: {pl.__version__}")

In [ ]:
# ============================================================
# 📁 FILE PATHS - UPDATE THESE TO MATCH YOUR DRIVE STRUCTURE
# ============================================================

DRIVE_BASE = "/content/drive/MyDrive"  # Your Google Drive root

# Target File (Main Training Set) - ~10M rows
TARGET_PATH = f"{DRIVE_BASE}/data/processed/train_balanced_xgboost.parquet"

# Source File (Raw Metadata) - ~132M rows
SOURCE_PATH = f"{DRIVE_BASE}/data/raw/metabonet_public_train.parquet"

# Output File
OUTPUT_PATH = f"{DRIVE_BASE}/data/processed/train_augmented.parquet"

# Columns to extract from Source File
SOURCE_COLUMNS = ['id', 'date', 'basal', 'age', 'weight', 'gender', 'insulin_type_bolus']

print("📁 File paths configured:")
print(f"   Target: {TARGET_PATH}")
print(f"   Source: {SOURCE_PATH}")
print(f"   Output: {OUTPUT_PATH}")

## 2. Memory-Efficient Data Loading

In [ ]:
# Load Target File (Eager - acceptable for ~10M rows)
print("📥 Loading Target File (eager mode)...")
target_df = pl.read_parquet(TARGET_PATH)

print(f"✅ Target loaded: {target_df.shape[0]:,} rows x {target_df.shape[1]} columns")
print(f"   Memory usage: ~{target_df.estimated_size('mb'):.1f} MB")
print(f"\n📊 Target Schema:")
print(target_df.schema)

In [ ]:
# Load Source File (Lazy Mode - CRITICAL for 132M rows)
print("📥 Scanning Source File (lazy mode)...")

# OPTIMIZATION: Select only required columns immediately to minimize I/O
source_lazy = (
    pl.scan_parquet(SOURCE_PATH)
    .select(SOURCE_COLUMNS)  # Only load needed columns
)

print("✅ Source LazyFrame created (no data loaded yet)")
print(f"\n📊 Source Schema (selected columns only):")
print(source_lazy.collect_schema())

## 3. Data Type Alignment (Critical for Joining)

In [ ]:
# Check date types in both dataframes
target_date_type = target_df.schema.get('date')
source_schema = source_lazy.collect_schema()
source_date_type = source_schema.get('date')

print("🔍 Date Type Analysis:")
print(f"   Target 'date' type: {target_date_type}")
print(f"   Source 'date' type: {source_date_type}")

# Check id types
target_id_type = target_df.schema.get('id')
source_id_type = source_schema.get('id')

print(f"\n   Target 'id' type: {target_id_type}")
print(f"   Source 'id' type: {source_id_type}")

In [ ]:
def align_data_types(target_df: pl.DataFrame, source_lazy: pl.LazyFrame) -> tuple:
    """
    Align data types between target and source for successful joining.
    
    Handles:
    - Int64 timestamp -> Datetime conversion
    - String/Categorical id alignment
    """
    target_schema = target_df.schema
    source_schema = source_lazy.collect_schema()
    
    # ========== DATE ALIGNMENT ==========
    target_date_type = target_schema.get('date')
    source_date_type = source_schema.get('date')
    
    if source_date_type == pl.Int64 and target_date_type in [pl.Datetime, pl.Datetime('ms'), pl.Datetime('ns'), pl.Datetime('us')]:
        print("⚙️ Converting Source 'date': Int64 -> Datetime (milliseconds)")
        source_lazy = source_lazy.with_columns(
            pl.col('date').cast(pl.Datetime('ms')).alias('date')
        )
    elif source_date_type != target_date_type:
        # Try to cast source to match target
        print(f"⚙️ Converting Source 'date': {source_date_type} -> {target_date_type}")
        source_lazy = source_lazy.with_columns(
            pl.col('date').cast(target_date_type).alias('date')
        )
    else:
        print("✅ Date types already aligned")
    
    # ========== ID ALIGNMENT ==========
    target_id_type = target_schema.get('id')
    source_id_type = source_schema.get('id')
    
    if source_id_type != target_id_type:
        # Convert source id to match target
        print(f"⚙️ Converting Source 'id': {source_id_type} -> {target_id_type}")
        source_lazy = source_lazy.with_columns(
            pl.col('id').cast(target_id_type).alias('id')
        )
    else:
        print("✅ ID types already aligned")
    
    return target_df, source_lazy


# Apply alignment
target_df, source_lazy = align_data_types(target_df, source_lazy)
print("\n✅ Data types aligned successfully!")

## 4. Perform the Join (Left Join)

In [ ]:
# Determine which columns are new from source (excluding join keys)
join_keys = ['id', 'date']
new_columns = [col for col in SOURCE_COLUMNS if col not in join_keys]

print(f"🔗 Join Configuration:")
print(f"   Join Keys: {join_keys}")
print(f"   New Columns to Add: {new_columns}")

In [ ]:
%%time

print("🔄 Performing LEFT JOIN...")
print("   (This may take several minutes for large datasets)")

# Convert target to lazy for optimized join
target_lazy = target_df.lazy()

# Perform the join
# Using suffix to handle any potential column name conflicts
augmented_lazy = target_lazy.join(
    source_lazy,
    on=join_keys,
    how='left',
    suffix='_source'  # Suffix for duplicate columns (if any)
)

# Collect the result
print("📊 Collecting joined data...")
augmented_df = augmented_lazy.collect()

# Clear memory
del target_lazy
gc.collect()

print(f"\n✅ Join completed!")
print(f"   Result: {augmented_df.shape[0]:,} rows x {augmented_df.shape[1]} columns")
print(f"   Memory: ~{augmented_df.estimated_size('mb'):.1f} MB")

In [ ]:
# Check for null values in new columns (unmatched rows)
print("📊 Null Value Analysis (new columns):")
for col in new_columns:
    if col in augmented_df.columns:
        null_count = augmented_df[col].null_count()
        null_pct = (null_count / len(augmented_df)) * 100
        print(f"   {col}: {null_count:,} nulls ({null_pct:.2f}%)")

## 5. Feature Engineering

In [ ]:
print("⚙️ Applying Feature Engineering...")

# Check what columns we have
print(f"\n📊 Current columns: {augmented_df.columns}")

In [ ]:
# Feature Engineering transformations
transformations = []

# ----- GENDER ENCODING -----
# Map 'M' -> 0, 'F' -> 1
if 'gender' in augmented_df.columns:
    print("\n🔢 Gender Encoding: 'M' -> 0, 'F' -> 1")
    print(f"   Unique values before: {augmented_df['gender'].unique().to_list()}")
    
    transformations.append(
        pl.when(pl.col('gender') == 'M')
          .then(pl.lit(0))
          .when(pl.col('gender') == 'F')
          .then(pl.lit(1))
          .otherwise(pl.lit(None))
          .cast(pl.Int8)
          .alias('gender_encoded')
    )

# ----- INSULIN TYPE LABEL ENCODING -----
if 'insulin_type_bolus' in augmented_df.columns:
    print("\n🔢 Insulin Type Label Encoding")
    unique_insulin_types = augmented_df['insulin_type_bolus'].drop_nulls().unique().sort().to_list()
    print(f"   Unique insulin types: {unique_insulin_types}")
    
    # Create mapping dictionary
    insulin_mapping = {v: i for i, v in enumerate(unique_insulin_types)}
    print(f"   Mapping: {insulin_mapping}")
    
    # Apply label encoding using replace
    transformations.append(
        pl.col('insulin_type_bolus')
          .replace(insulin_mapping, default=None)
          .cast(pl.Int16)
          .alias('insulin_type_encoded')
    )

# ----- CAST NUMERIC COLUMNS TO FLOAT32 -----
float_cols = ['basal', 'age', 'weight']
for col in float_cols:
    if col in augmented_df.columns:
        print(f"\n🔢 Casting '{col}' to Float32")
        transformations.append(
            pl.col(col).cast(pl.Float32).alias(col)
        )

# Apply all transformations
if transformations:
    augmented_df = augmented_df.with_columns(transformations)
    print("\n✅ Feature engineering completed!")

In [ ]:
# Verify the new columns
print("\n📊 Updated Schema:")
for col, dtype in augmented_df.schema.items():
    print(f"   {col}: {dtype}")

## 6. Save Results

In [ ]:
%%time

print(f"💾 Saving augmented dataset to:")
print(f"   {OUTPUT_PATH}")

# Save with compression for smaller file size
augmented_df.write_parquet(
    OUTPUT_PATH,
    compression='zstd',  # Good balance of speed and compression
    compression_level=3
)

print("\n✅ File saved successfully!")

In [ ]:
# Get file size
import os

file_size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
print(f"📁 Output file size: {file_size_mb:.1f} MB")

## 7. Verification

In [ ]:
# Final verification
print("="*60)
print("📊 FINAL RESULT SUMMARY")
print("="*60)

print(f"\n📐 Shape: {augmented_df.shape[0]:,} rows x {augmented_df.shape[1]} columns")

print(f"\n📋 Columns ({len(augmented_df.columns)}):")
for i, col in enumerate(augmented_df.columns, 1):
    print(f"   {i:2}. {col}")

In [ ]:
# Show first 5 rows
print("\n📝 First 5 rows:")
augmented_df.head(5)

In [ ]:
# Verify new metadata columns are present
print("\n🔍 Verification - New Columns Added:")
expected_new_cols = ['basal', 'age', 'weight', 'gender', 'gender_encoded', 
                     'insulin_type_bolus', 'insulin_type_encoded']

for col in expected_new_cols:
    status = "✅" if col in augmented_df.columns else "❌"
    print(f"   {status} {col}")

In [ ]:
# Statistics for numeric columns
print("\n📈 Statistics for New Numeric Columns:")
numeric_cols = ['basal', 'age', 'weight', 'gender_encoded', 'insulin_type_encoded']
existing_numeric = [c for c in numeric_cols if c in augmented_df.columns]

if existing_numeric:
    augmented_df.select(existing_numeric).describe()

In [ ]:
# Cleanup
del augmented_df
gc.collect()

print("\n🎉 Data Augmentation Pipeline Complete!")
print(f"   Output saved to: {OUTPUT_PATH}")